### Calculating everything needed as tarHMM input
- all pretrained SAM3 tracks on the gt crops; t cell stats only
- Use AnalysisEnv

In [1]:
import pickle
import numpy as np

import zipfile
from io import BytesIO
import tifffile

import os

from pathlib import Path
import yaml

In [2]:
cvat_base_dir = '/gladstone/engelhardt/lab/MarsonLabIncucyteData/UltrackAnalysis/groundTruthTracks/TCR-T/'
SAM3_base_dir = "/gladstone/engelhardt/lab/jutran/lci/MarsonImagingPipeline/data/cell_type_assignment/original_SAM3_track_assignments/"

crop_ids = ['B4_t50t100y200y350x750x900',
            'B8_t50t100y200y350x750x900',
            'E4_t50t100y200y350x750x900',
            'B4_t250t300y200y350x750x900',
            'B8_t250t300y200y350x750x900',
            'E4_t250t300y200y350x750x900']

def filter_tracks(track_type, tracks, type_dict):
    """Filter tracks to only contain tracks of the desired type

    Args:
        track_type (str): Type of track to filter for, matching a key in type_dict
        tracks (np.ndarray): Tracked cell mask [T, Y, X]
        type_dict (dict): Dictionary mapping cell id to cell type
    """

    valid_ids = [cell_id for cell_id, cell_type in type_dict.items() if cell_type == track_type]

    filtered_tracks = tracks.copy()
    filtered_tracks[~np.isin(filtered_tracks, valid_ids)] = 0
    
    return filtered_tracks

### Calculate masking metadata

In [8]:
sam3_tracks_per_well = {}
t_cell_tracks_per_well = {}
cancer_tracks_per_well = {}

for crop in crop_ids:
    well_id = crop.split('_')[0]

    # load all tracks 
    sam3_tracks = tifffile.imread(os.path.join(SAM3_base_dir, crop, f'reindexed_tracks.tiff'))
    sam3_tracks_per_well[crop] = sam3_tracks

    cell_type_dict = pickle.load(open(os.path.join(SAM3_base_dir, crop, "full_cell_type_dict.pkl"), "rb"))
    t_cell_tracks_per_well[crop] = filter_tracks("t_cell", sam3_tracks, cell_type_dict)
    cancer_tracks_per_well[crop] = filter_tracks("cancer", sam3_tracks, cell_type_dict)


In [37]:
# calculate actual statistics
data = {}

# Initialize all masks
active_mask = np.zeros([50,0], dtype=bool)
is_division_mask = np.zeros([50,0], dtype=bool)
is_new_root_mask = np.zeros([50,0], dtype=bool)
parent_indices = np.zeros([50,0], dtype=np.int32)

for crop in crop_ids:
    t_cell_tracks = t_cell_tracks_per_well[crop]

    T = t_cell_tracks.shape[0]
    all_cell_ids = np.unique(t_cell_tracks[t_cell_tracks > 0])
    all_cell_ids.sort()
    num_cells = len(all_cell_ids)
    id_to_col = {int(cid): i for i, cid in enumerate(all_cell_ids)}

    print(f"crop={crop}, T={T}, num_cells={num_cells}")

    # Initialize all masks
    crop_active_mask = np.zeros((T, num_cells), dtype=bool)
    crop_is_division_mask = np.zeros((T, num_cells), dtype=bool)
    crop_is_new_root_mask = np.zeros((T, num_cells), dtype=bool)
    crop_parent_indices = np.zeros((T, num_cells), dtype=np.int32)

    # Make active_mask from t_cell_tracks
    for t in range(T):
        t_cell_frame = t_cell_tracks[t]
        frame_ids = np.unique(t_cell_frame[t_cell_frame > 0])
        for cid in frame_ids:
            crop_active_mask[t, id_to_col[int(cid)]] = True

    # 4. Build is_new_root_mask, parent_indices (keep is_division_mask as all False)
    for cid, col in id_to_col.items():
        active_frames = np.where(crop_active_mask[:, col])[0]
        if len(active_frames) == 0:
            print(f"Warning: Cell ID {cid} (col {col}) is never active in active_mask. Skipping.")
            continue

        first_frame = active_frames[0]

        # This cell is a root (appeared spontaneously or is the initial cell)
        crop_is_new_root_mask[first_frame, col] = True
        crop_parent_indices[first_frame, col] = col  # self

        # For all subsequent active frames, parent = self (cell continues)
        for t in active_frames[1:]:
            crop_parent_indices[t, col] = col

    # Append crop masks to overall masks
    active_mask = np.concatenate((active_mask, crop_active_mask), axis=1)
    is_division_mask = np.concatenate((is_division_mask, crop_is_division_mask), axis=1)
    is_new_root_mask = np.concatenate((is_new_root_mask, crop_is_new_root_mask), axis=1)
    parent_indices = np.concatenate((parent_indices, crop_parent_indices), axis=1)

    # 5. Summary
    print(f"active_mask shape:       {active_mask.shape}")
    print(f"parent_indices shape:    {parent_indices.shape}")
    print(f"is_division_mask shape:  {is_division_mask.shape}")
    print(f"is_new_root_mask shape:  {is_new_root_mask.shape}")
    print(f"Division events (daughters): {is_division_mask.sum()}")
    print(f"Root cells:                  {is_new_root_mask.sum()}")

data['active_mask'] = active_mask
data['is_division_mask'] = is_division_mask
data['is_new_root_mask'] = is_new_root_mask
data['parent_indices'] = parent_indices

crop=B4_t50t100y200y350x750x900, T=50, num_cells=57
active_mask shape:       (50, 57)
parent_indices shape:    (50, 57)
is_division_mask shape:  (50, 57)
is_new_root_mask shape:  (50, 57)
Division events (daughters): 0
Root cells:                  57
crop=B8_t50t100y200y350x750x900, T=50, num_cells=77
active_mask shape:       (50, 134)
parent_indices shape:    (50, 134)
is_division_mask shape:  (50, 134)
is_new_root_mask shape:  (50, 134)
Division events (daughters): 0
Root cells:                  134
crop=E4_t50t100y200y350x750x900, T=50, num_cells=22
active_mask shape:       (50, 156)
parent_indices shape:    (50, 156)
is_division_mask shape:  (50, 156)
is_new_root_mask shape:  (50, 156)
Division events (daughters): 0
Root cells:                  156
crop=B4_t250t300y200y350x750x900, T=50, num_cells=55
active_mask shape:       (50, 211)
parent_indices shape:    (50, 211)
is_division_mask shape:  (50, 211)
is_new_root_mask shape:  (50, 211)
Division events (daughters): 0
Root cells:  

### Calculate feature values

In [32]:
os.chdir("/gladstone/engelhardt/lab/jutran/lci/MarsonImagingPipeline")
from scripts.utils.StatUtils import *

In [33]:
# calculate emissions feature statistics

# calculate t cell velocities
t_cell_velocities_per_frame = {well: compute_cell_velocities_per_frame_dict(t_cell_tracks_per_well[well], unit_per_frame=1) for well in t_cell_tracks_per_well.keys()}


Computing cell velocities: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████| 49/49 [00:00<00:00, 1367.11it/s]


In [34]:
# calculate type-specific interactions

cancer_type_specific_contacts_per_frame, t_cell_type_specific_contacts_per_frame = {}, {}
cancer_type_specific_neighbors_per_frame, t_cell_type_specific_neighbors_per_frame = {}, {}

for well in sam3_tracks_per_well.keys():
    t_cell_tracks = t_cell_tracks_per_well[well]
    cancer_tracks = cancer_tracks_per_well[well]

    well_cancer_contacts_per_frame, well_t_cell_contacts_per_frame = compute_cell_cell_contact_dict(t_cell_tracks, cancer_tracks)
    well_cancer_neighbors_per_frame, well_t_cell_neighbors_per_frame = compute_cell_cell_neighbor_dict(t_cell_tracks, cancer_tracks)

    cancer_type_specific_contacts_per_frame[well] = well_cancer_contacts_per_frame
    t_cell_type_specific_contacts_per_frame[well] = well_t_cell_contacts_per_frame

    cancer_type_specific_neighbors_per_frame[well] = well_cancer_neighbors_per_frame
    t_cell_type_specific_neighbors_per_frame[well] = well_t_cell_neighbors_per_frame

Computing cell-cell contact dataframe...


Processing frame-by-frame contact output: 100%|█████████████████████████████████████████████████████████████████████████████████████████████| 50/50 [00:00<00:00, 168.63it/s]


Computing cell-cell neighbor dataframe...


Processing frame-by-frame neighbor output: 100%|████████████████████████████████████████████████████████████████████████████████████████████| 50/50 [00:00<00:00, 227.80it/s]


Computing cell-cell contact dataframe...


...

Processing frame-by-frame neighbor output: 100%|████████████████████████████████████████████████████████████████████████████████████████████| 50/50 [00:00<00:00, 295.85it/s]


Computing cell-cell contact dataframe...


Processing frame-by-frame contact output: 100%|█████████████████████████████████████████████████████████████████████████████████████████████| 50/50 [00:00<00:00, 178.68it/s]


Computing cell-cell neighbor dataframe...


Processing frame-by-frame neighbor output: 100%|████████████████████████████████████████████████████████████████████████████████████████████| 50/50 [00:00<00:00, 246.75it/s]


In [35]:
# calculate type-agnostic interactions

cancer_all_contacts_per_frame, t_cell_all_contacts_per_frame = {}, {}
cancer_all_neighbors_per_frame, t_cell_all_neighbors_per_frame = {}, {}

for well in sam3_tracks_per_well.keys():
    t_cell_tracks = t_cell_tracks_per_well[well]
    cancer_tracks = cancer_tracks_per_well[well]

    well_cancer_contacts_per_frame, well_t_cell_contacts_per_frame = compute_all_cell_cell_contact_dict(t_cell_tracks, cancer_tracks)
    well_cancer_neighbors_per_frame, well_t_cell_neighbors_per_frame = compute_all_cell_cell_neighbor_dict(t_cell_tracks, cancer_tracks)

    cancer_all_contacts_per_frame[well] = well_cancer_contacts_per_frame
    t_cell_all_contacts_per_frame[well] = well_t_cell_contacts_per_frame

    cancer_all_neighbors_per_frame[well] = well_cancer_neighbors_per_frame
    t_cell_all_neighbors_per_frame[well] = well_t_cell_neighbors_per_frame

Computing cell-cell contact dataframe...


Processing contact dict: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████| 50/50 [00:00<00:00, 140.44it/s]


Computing cell-cell neighbor dataframe...


Processing neighbor dict: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████| 50/50 [00:00<00:00, 154.00it/s]


Computing cell-cell contact dataframe...


...

Processing neighbor dict: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████| 50/50 [00:00<00:00, 126.02it/s]


Computing cell-cell contact dataframe...


Processing contact dict: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████| 50/50 [00:00<00:00, 121.67it/s]


Computing cell-cell neighbor dataframe...


Processing neighbor dict: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████| 50/50 [00:00<00:00, 132.17it/s]


### Create emissions array

In [36]:
emissions_array = np.zeros((50, 0, 3)) # shape: (num_frames, num_t_cells, num_emission_features)

for crop in crop_ids:
    t_cell_tracks = t_cell_tracks_per_well[crop]
    t_cell_velocities = t_cell_velocities_per_frame[crop]
    t_cell_type_specific_neighbors = t_cell_type_specific_neighbors_per_frame[crop]
    t_cell_all_neighbors = t_cell_all_neighbors_per_frame[crop]

    T = t_cell_tracks.shape[0]
    t_cell_ids = np.unique(t_cell_tracks[t_cell_tracks > 0])
    t_cell_ids.sort()
    id_to_column_index = {cell_id: index for index, cell_id in enumerate(t_cell_ids)}
    
    print(f"crop={crop}, T={T}, num_cells={len(t_cell_ids)}")

    crop_emissions = np.zeros((T, len(t_cell_ids), 3)) # 3 features: velocity, type-specific neighbors, all neighbors

    for t in range(T):
        frame = t_cell_tracks[t]
        for cell_id in np.unique(frame[frame > 0]):
            column_index = id_to_column_index[cell_id]

            if t > 0:
                if cell_id in t_cell_velocities[t]:
                    velocity = t_cell_velocities[t][cell_id]
                    crop_emissions[t, column_index, 0] = velocity

            if cell_id in t_cell_all_neighbors[t]:
                all_neighbors = t_cell_all_neighbors[t][cell_id]
                cancer_neighbors_list = t_cell_type_specific_neighbors[t].get(cell_id, [0])
                cancer_neighbors = cancer_neighbors_list[0]

                t_cell_neighbors = all_neighbors - cancer_neighbors

                crop_emissions[t, column_index, 1] = cancer_neighbors
                crop_emissions[t, column_index, 2] = t_cell_neighbors

    emissions_array = np.concatenate((emissions_array, crop_emissions), axis=1)
    print(f"emissions_array shape after processing {crop}: {emissions_array.shape}")

crop=B4_t50t100y200y350x750x900, T=50, num_cells=57
emissions_array shape after processing B4_t50t100y200y350x750x900: (50, 57, 3)
crop=B8_t50t100y200y350x750x900, T=50, num_cells=77
emissions_array shape after processing B8_t50t100y200y350x750x900: (50, 134, 3)
crop=E4_t50t100y200y350x750x900, T=50, num_cells=22
emissions_array shape after processing E4_t50t100y200y350x750x900: (50, 156, 3)
crop=B4_t250t300y200y350x750x900, T=50, num_cells=55
emissions_array shape after processing B4_t250t300y200y350x750x900: (50, 211, 3)
crop=B8_t250t300y200y350x750x900, T=50, num_cells=88
emissions_array shape after processing B8_t250t300y200y350x750x900: (50, 299, 3)
crop=E4_t250t300y200y350x750x900, T=50, num_cells=27
emissions_array shape after processing E4_t250t300y200y350x750x900: (50, 326, 3)


### Generate crop-specific array col idx mapping

In [38]:
all_crop_cell_ids = []

for crop in crop_ids:
    t_cell_tracks = t_cell_tracks_per_well[crop]
    t_cell_ids = np.unique(t_cell_tracks[t_cell_tracks > 0])
    t_cell_ids.sort()

    crop_t_cell_ids = [f"{crop}_{cell_id}" for cell_id in t_cell_ids]
    all_crop_cell_ids.extend(crop_t_cell_ids)

id_to_col = {cid: i for i, cid in enumerate(all_crop_cell_ids)}

### save data

In [40]:
output_dir = "/gladstone/engelhardt/lab/jutran/lci/treeHMM/notebooks/data/all_gt_crops_pretrained_SAM3/"

with open(output_dir + "data.pkl", "wb") as f:
    pickle.dump(data, f)

with open(output_dir + "crop_and_cell_id_to_col.pkl", "wb") as f:
    pickle.dump(id_to_col, f)

np.save(output_dir + "emissions_array.npy", emissions_array)